In [1]:
!pip install -q transformers accelerate

In [2]:
import polars as pl
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer
from collections import Counter

In [3]:
MODEL_NAME = "microsoft/phi-2"

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    torch_dtype=torch.float16,
    device_map="auto"
)

print("Model loaded!")

config.json:   0%|          | 0.00/735 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

added_tokens.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/99.0 [00:00<?, ?B/s]

...

model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/453 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

Model loaded!


In [4]:
test = pl.read_csv('/kaggle/input/nvidia-nemotron-3-reasoning-challenge/test.csv')

In [5]:
def build_prompt(prompt):
    return f"""
You are a reasoning expert.

Solve the problem carefully using the examples.
Give ONLY the final answer.

{prompt}

Final Answer:
"""

In [6]:
def get_answer(prompt):
    full_prompt = build_prompt(prompt)
    
    inputs = tokenizer(full_prompt, return_tensors="pt").to(model.device)
    
    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=80,
            do_sample=False   # deterministic = faster
        )
    
    response = tokenizer.decode(outputs[0], skip_special_tokens=True)
    
    if "Final Answer:" in response:
        return response.split("Final Answer:")[-1].strip()
    
    return response.strip()

In [7]:
predictions = []

for i, row in enumerate(test.iter_rows(named=True)):
    prompt = row["prompt"]
    
    pred = get_answer(prompt)
    predictions.append(pred)
    
    if i % 10 == 0:
        print(f"Done {i}")

Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.


Done 0


Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.


In [8]:
submission = pl.DataFrame({
    "id": test["id"],
    "output": predictions
})

submission.write_csv("submission.csv")

print("Submission ready!")

Submission ready!


In [9]:
import os
import zipfile
import torch
from transformers import AutoModelForCausalLM
from peft import LoraConfig, get_peft_model, TaskType

# Load SMALL model (fast)
model = AutoModelForCausalLM.from_pretrained(
    "microsoft/phi-2",
    torch_dtype=torch.float16,
    device_map="cpu"
)

# Create LoRA config
lora_config = LoraConfig(
    r=4,
    lora_alpha=8,
    target_modules=["q_proj", "v_proj"],
    lora_dropout=0.0,
    bias="none",
    task_type=TaskType.CAUSAL_LM,
)

# Apply LoRA (this creates REAL weights)
model = get_peft_model(model, lora_config)

# Save adapter properly
save_path = "/kaggle/working/adapter"
model.save_pretrained(save_path)

# Zip ONLY required files
zip_path = "/kaggle/working/submission.zip"

with zipfile.ZipFile(zip_path, "w") as z:
    for file in os.listdir(save_path):
        z.write(os.path.join(save_path, file), arcname=file)

print("FILES:", os.listdir("/kaggle/working"))

Loading weights:   0%|          | 0/453 [00:00<?, ?it/s]

/usr/local/lib/python3.12/dist-packages/torchao/float8/float8_tensor.py:122: FutureWarning: torch._dynamo.allow_in_graph is deprecated and will be removed in a future version. Use torch._dynamo.nonstrict_trace instead.
  @torch._dynamo.allow_in_graph
/usr/local/lib/python3.12/dist-packages/torchao/float8/float8_tensor.py:195: FutureWarning: torch._dynamo.allow_in_graph is deprecated and will be removed in a future version. Use torch._dynamo.nonstrict_trace instead.
  @torch._dynamo.allow_in_graph
/usr/local/lib/python3.12/dist-packages/torchao/float8/float8_scaling_utils.py:90: FutureWarning: torch._dynamo.allow_in_graph is deprecated and will be removed in a future version. Use torch._dynamo.nonstrict_trace instead.
  @torch._dynamo.allow_in_graph
/usr/local/lib/python3.12/dist-packages/torchao/float8/float8_linear.py:60: FutureWarning: torch._dynamo.allow_in_graph is deprecated and will be removed in a future version. Use torch._dynamo.nonstrict_trace instead.
  @torch._dynamo.allow_

FILES: ['adapter', '__notebook__.ipynb', 'submission.zip', 'submission.csv']
